# Clustering: K-Means, GMM, DBSCAN & Hierarchical

This notebook walks through the from-scratch implementations in the `clustering` package:

1. Load four labeled synthetic datasets
2. Fit **K-Means**, **Soft K-Means**, **GMM (EM)**, **DBSCAN**, and **Ward agglomerative**
3. Compare against scikit-learn
4. Choose \(k\) with silhouette / BIC

See `docs/algorithms.md` for the math and design notes.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans as SkKMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture as SkGaussianMixture

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from clustering import (
    AgglomerativeClustering,
    DBSCAN,
    GaussianMixtureModel,
    KMeans,
    SoftKMeans,
    clustering_report,
)
from clustering.model_selection import (
    elbow_curve,
    gmm_information_criteria,
    suggest_k,
)
from clustering.viz import plot_datasets, plot_kmeans_comparison, plot_soft_assignments

%matplotlib inline


## 1. Datasets

Each CSV has columns `x`, `y`, and ground-truth `cat` (3 classes, 1000 points each).

In [ ]:
names = ["mv", "unif", "mv2", "mv3"]
dataframes = [pd.read_csv(ROOT / "data" / f"{name}.csv", index_col=0) for name in names]
datasets = {
    name: (df[["x", "y"]].to_numpy(dtype=float), df["cat"].to_numpy())
    for name, df in zip(names, dataframes)
}

fig = plot_datasets(dataframes, titles=names)
plt.show()

## 2. Custom K-Means vs scikit-learn

Centers from the custom estimator (black ×) should land next to sklearn centers (red +).

In [ ]:
fig = plot_kmeans_comparison(dataframes, n_clusters=3, random_state=0)
plt.show()

rows = []
for name, (X, y) in datasets.items():
    custom = KMeans(n_clusters=3, random_state=0).fit(X)
    sk = SkKMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
    rows.append({
        "dataset": name,
        "custom_ari": adjusted_rand_score(y, custom.labels_),
        "sklearn_ari": adjusted_rand_score(y, sk.labels_),
        "custom_inertia": custom.inertia_,
        "sklearn_inertia": sk.inertia_,
    })
pd.DataFrame(rows)

## 3. Soft K-Means and GMM

Soft methods shine when clusters overlap or stretch. On `mv2`, GMM typically beats hard K-Means by a wide ARI margin.

In [ ]:
rows = []
for name, (X, y) in datasets.items():
    soft = SoftKMeans(n_clusters=3, beta=0.3, random_state=0).fit(X)
    gmm = GaussianMixtureModel(n_components=3, random_state=0).fit(X)
    sk_gmm = SkGaussianMixture(n_components=3, random_state=0).fit(X)
    rows.append({
        "dataset": name,
        "soft_kmeans_ari": adjusted_rand_score(y, soft.labels_),
        "custom_gmm_ari": adjusted_rand_score(y, gmm.labels_),
        "sklearn_gmm_ari": adjusted_rand_score(y, sk_gmm.predict(X)),
        "custom_gmm_bic": gmm.bic(X),
    })
pd.DataFrame(rows)

In [ ]:
fig = plot_soft_assignments(dataframes, model="gmm", n_clusters=3, random_state=0)
plt.show()
fig = plot_soft_assignments(dataframes, model="kmeans", n_clusters=3, random_state=0)
plt.show()

## 4. Choosing the number of clusters

Use silhouette for K-Means and BIC for GMMs.

In [ ]:
X, y = datasets["mv3"]
elbow = pd.DataFrame(elbow_curve(X, range(1, 8), random_state=0))
ics = pd.DataFrame(gmm_information_criteria(X, range(1, 8), random_state=0))

print("Suggested k (silhouette):", suggest_k(elbow.to_dict("records"), "silhouette"))
print("Suggested k (BIC):", suggest_k(ics.to_dict("records"), "bic"))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(elbow["k"], elbow["inertia"], marker="o")
axes[0].set_title("K-Means elbow (inertia)")
axes[0].set_xlabel("k")
axes[1].plot(ics["k"], ics["bic"], marker="o", label="BIC")
axes[1].plot(ics["k"], ics["aic"], marker="s", label="AIC")
axes[1].set_title("GMM information criteria")
axes[1].set_xlabel("k")
axes[1].legend()
plt.tight_layout()
plt.show()

display(elbow)
display(ics)

## 5. Single-dataset report

Compact metrics for portfolio / debugging use.

In [ ]:
X, y = datasets["mv2"]
km = KMeans(n_clusters=3, random_state=0).fit(X)
gmm = GaussianMixtureModel(n_components=3, random_state=0).fit(X)

print("K-Means", clustering_report(X, y, km.labels_, km.cluster_centers_))
print("GMM", clustering_report(X, y, gmm.labels_, gmm.means_))

## Extra: DBSCAN and hierarchical clustering

On well-separated `mv3`, density and Ward linkage should also recover the three blobs.

In [ ]:
X, y = datasets["mv3"]
db = DBSCAN(eps=0.35, min_samples=12).fit(X)
agg = AgglomerativeClustering(n_clusters=3, linkage="ward").fit(X)
print({
    "dbscan_clusters": db.n_clusters_,
    "dbscan_ari": adjusted_rand_score(y, db.labels_),
    "ward_ari": adjusted_rand_score(y, agg.labels_),
})
